# Clase 6 — Exploración, KNN y clases desbalanceadas

## Pregunta central

> ¿Qué puede salir mal si un modelo ve muchos ejemplos de una clase y muy pocos de la otra?

En las clases anteriores vimos que un proyecto de Machine Learning comienza con una pregunta y continúa con la exploración de los datos, la preparación, el entrenamiento y la evaluación.

Hoy vamos a recorrer ese flujo con un dataset real de solicitudes de crédito. La exploración revelará un problema común: la variable objetivo no está balanceada. Primero entrenaremos un clasificador sencillo de vecinos cercanos y observaremos su sesgo. Después probaremos una posible respuesta: generar ejemplos sintéticos de la clase minoritaria dentro del conjunto de entrenamiento.

pregunta → exploración → preparación → train/test → KNN → evaluación → mejora


## Objetivos

Al finalizar deberías poder:

- recorrer un flujo básico de clasificación con datos tabulares reales;
- usar una exploración inicial para detectar tipos de datos, faltantes y desbalance;
- preparar variables numéricas para un modelo basado en distancias;
- interpretar una matriz de confusión más allá de la accuracy;
- explicar cómo el desbalance puede favorecer la clase mayoritaria;
- generar ejemplos sintéticos y evaluar si el cambio ayuda.

## Cómo trabajar

1. Ejecutá las celdas en orden.
2. Observá los resultados antes de leer la interpretación.
3. En la actividad final modificá solo las variables señaladas.
4. El foco está en conectar decisiones sobre datos con resultados del modelo.


---
## 1. Repaso: el proceso de Machine Learning

Un modelo no es el primer paso.

| Paso | Pregunta que guía la decisión |
|---|---|
| Definir la tarea | ¿Qué queremos predecir y para qué? |
| Explorar | ¿Qué columnas hay? ¿Qué problemas tienen? |
| Preparar | ¿Qué columnas usaremos y en qué escala? |
| Separar | ¿Cómo mediremos con datos que no vimos? |
| Entrenar | ¿Qué patrón aprende el modelo? |
| Evaluar | ¿Qué errores comete y cuánto cuestan? |
| Mejorar | ¿Qué hipótesis podemos probar? |

La clase de hoy empieza en explorar. El problema de desbalance aparecerá antes de entrenar.


---
## 2. Cargar el dataset de crédito

El archivo contiene solicitudes de crédito. La columna objetivo es moroso:

- 0 significa que la solicitud no fue clasificada como morosa;
- 1 significa que fue clasificada como morosa.

No vamos a construir un sistema de crédito para producción. Usaremos el archivo para aprender a leer un dataset tabular, detectar un problema y comparar decisiones de modelado.

El CSV se encuentra dentro de Modulo 1 para que el notebook sea reproducible junto con la clase.


In [ ]:
# ------------------------------------------------------------
# PREPARACIÓN DEL NOTEBOOK
# ------------------------------------------------------------
# No hace falta memorizar estas importaciones.

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42

# Funciona tanto si Jupyter se abre desde Modulo 1 como desde la raíz del proyecto.
rutas_posibles = [
    Path("dataset_credito.csv"),
    Path("Modulo 1") / "dataset_credito.csv",
]
ruta_dataset = next((ruta for ruta in rutas_posibles if ruta.exists()), None)
if ruta_dataset is None:
    raise FileNotFoundError(
        "No se encontró dataset_credito.csv. Abrí el notebook desde el proyecto."
    )

creditos = pd.read_csv(ruta_dataset)
print("Archivo:", ruta_dataset)
print("Forma del dataset (filas, columnas):", creditos.shape)
display(creditos.head())


### Exploración inicial: estructura y calidad

Antes de elegir variables o modelos, miramos tipos, faltantes, duplicados y rangos. El objetivo no es estudiar cada columna en profundidad; es encontrar riesgos que cambien nuestras decisiones.


In [ ]:
print("Tipos de datos:")
display(creditos.dtypes.to_frame("tipo"))

print("Filas duplicadas exactas:", creditos.duplicated().sum())

faltantes = pd.DataFrame({
    "cantidad": creditos.isna().sum(),
    "porcentaje": (creditos.isna().mean() * 100).round(1),
})
print("Columnas con valores faltantes:")
display(faltantes[faltantes["cantidad"] > 0])

print("Resumen de columnas numéricas:")
display(creditos.select_dtypes(include="number").describe().T.round(2))


### Explorar la variable objetivo

La variable objetivo es la respuesta que queremos predecir. Si sus clases aparecen en cantidades muy distintas, la accuracy puede ocultar un modelo que casi nunca detecta la clase minoritaria.


In [ ]:
distribucion = creditos["moroso"].value_counts().sort_index()
resumen_objetivo = pd.DataFrame({
    "cantidad": distribucion.to_numpy(),
    "porcentaje": (distribucion.to_numpy() / len(creditos) * 100).round(1),
}, index=["no moroso (0)", "moroso (1)"])
display(resumen_objetivo)

accuracy_mayoria = (creditos["moroso"] == 0).mean()
print(f"Baseline ingenuo: si siempre digo 'no moroso', accuracy = {accuracy_mayoria:.1%}")


### Una vista rápida de dos variables

Para visualizar elegimos ingresos y endeudamiento externo. Son solo una proyección del dataset: el clasificador utilizará once variables numéricas.

Las columnas categóricas se dejan para una clase posterior, donde podremos codificarlas con cuidado.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colores = {0: "steelblue", 1: "tomato"}
leyendas = {0: "no moroso (0)", 1: "moroso (1)"}

for clase in [0, 1]:
    grupo = creditos[creditos["moroso"] == clase]
    ax.scatter(
        grupo["ingresos"],
        grupo["Endeudamiento_Externo"],
        color=colores[clase],
        alpha=0.65,
        s=28,
        label=leyendas[clase],
    )

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("ingresos (escala logarítmica)")
ax.set_ylabel("endeudamiento externo")
ax.set_title("Exploración: la clase morosa es minoritaria")
ax.legend()
plt.show()


### Lo que reveló el EDA

Ya tenemos un diagnóstico inicial:

- hay columnas numéricas y categóricas;
- hay valores faltantes;
- ID identifica una fila, pero no describe una solicitud;
- la variable objetivo tiene menos casos morosos;
- para KNN debemos completar faltantes y llevar las variables a una escala comparable.

El gráfico muestra dos variables, pero el KNN usará once columnas numéricas. Un modelo real todavía requeriría revisar qué información estaba disponible al momento de decidir y si alguna columna introduce fuga de información.


---
## 3. Preparar los datos y entrenar KNN

KNN clasifica una nueva fila observando sus vecinos más cercanos. Si hay muchos más ejemplos de una clase, es más probable que la mayoría de los vecinos pertenezca a esa clase.

El orden importa:

1. separamos train y test;
2. aprendemos la mediana de faltantes usando solo train;
3. aprendemos la escala usando solo train;
4. transformamos train y test con esas mismas recetas;
5. entrenamos KNN únicamente con train.

Así evitamos que el test filtre información hacia el entrenamiento.


In [ ]:
variables = [
    "ingresos",
    "edad",
    "BCRA_Peor_Situacion",
    "Cantidad_consultas_7_dias",
    "Compromisos_Mensual",
    "Endeudamiento_Externo",
    "canti_moras",
    "dias_atraso",
    "Cuota",
    "cant_cuotas",
    "Monto_Otorgado",
]

X = creditos[variables]
y = creditos["moroso"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=SEED,
)

# Pipeline
VECINOS_BASE = 7
modelo_base = Pipeline([
    ("imputador", SimpleImputer(strategy="median")),
    ("escalador", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=VECINOS_BASE)),
])
modelo_base.fit(X_train, y_train)
y_pred_base = modelo_base.predict(X_test)

# Reutilizamos exactamente el mismo preprocesamiento para el balanceo posterior.
preprocesamiento = modelo_base[:-1]
X_train_preparado = preprocesamiento.transform(X_train)
X_test_preparado = preprocesamiento.transform(X_test)

print(f"Train: {len(X_train)} filas; morosos: {int(y_train.sum())}")
print(f"Test:  {len(X_test)} filas; morosos: {int(y_test.sum())}")

In [ ]:
def calcular_metricas(y_real, y_predicha):
    return {
        "accuracy": accuracy_score(y_real, y_predicha),
        "precision": precision_score(y_real, y_predicha, zero_division=0),
        "recall": recall_score(y_real, y_predicha, zero_division=0),
        "F1": f1_score(y_real, y_predicha, zero_division=0),
        "morosos predichos": int(np.sum(y_predicha)),
    }


metricas_base = calcular_metricas(y_test, y_pred_base)
display(pd.Series(metricas_base, name="KNN sin balancear").to_frame())


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_base,
    display_labels=["no moroso", "moroso"],
    colorbar=False,
    cmap="Oranges",
    ax=ax,
)
ax.set_title(f"KNN sin balancear; accuracy {metricas_base['accuracy']:.1%}")
ax.set_xlabel("Predicho")
ax.set_ylabel("Real")
plt.show()


### Interpretar la matriz de confusión

La diagonal son los aciertos. Mirá especialmente la fila moroso:

- columna no moroso: morosos reales que el modelo dejó pasar, falsos negativos;
- columna moroso: morosos reales que el modelo encontró, verdaderos positivos.

La accuracy se parece a la del baseline de la clase mayoritaria, pero eso no significa que el modelo resuelva bien la tarea. El recall responde una pregunta más útil: de todos los morosos reales, ¿cuántos encontramos?

Esta es la primera señal de sesgo: el modelo resulta conservador al marcar la clase menos frecuente. No es que KNN tenga una intención; su vecindario refleja los datos con los que fue entrenado.


---
## 4. Generar datos sintéticos para reducir el desbalance

Una posible respuesta es darle al modelo más ejemplos de la clase minoritaria. No vamos a copiar filas idénticas; crearemos puntos entre dos morosos reales de train y les agregaremos una variación pequeña.

Esta es la idea básica de técnicas como SMOTE:

1. elegir dos ejemplos minoritarios;
2. interpolar sus valores;
3. agregar una variación controlada;
4. conservar la etiqueta moroso.

Las filas sintéticas se crean después del split y solo a partir de train. El test conserva datos reales para que siga siendo una evaluación honesta.

Los puntos se generan en el espacio estandarizado que recibe KNN. Como ese espacio no conserva unidades humanas, el gráfico los transforma de vuelta y muestra únicamente una proyección sobre ingresos y edad. El modelo utiliza las once variables.

En un proyecto real hay que validar con el dominio que una combinación sintética sea posible. No todo punto matemático representa una solicitud válida.


In [ ]:
def crear_morosos_sinteticos(X_entrenamiento, y_entrenamiento, cantidad, ruido=0.05):
    """Crea puntos entre morosos reales de train."""
    if cantidad == 0:
        return np.empty((0, X_entrenamiento.shape[1]))

    positivos = X_entrenamiento[np.asarray(y_entrenamiento) == 1]
    if len(positivos) < 2:
        raise ValueError("Se necesitan al menos dos morosos en train.")

    generador = np.random.default_rng(SEED + cantidad)
    indice_a = generador.integers(0, len(positivos), size=cantidad)
    indice_b = generador.integers(0, len(positivos), size=cantidad)
    proporcion = generador.random((cantidad, 1))

    puntos_intermedios = (
        positivos[indice_a] * proporcion
        + positivos[indice_b] * (1 - proporcion)
    )
    return puntos_intermedios + generador.normal(
        0, ruido, size=puntos_intermedios.shape
    )


# Calculamos cuántos morosos faltan para igualar las clases dentro de train.
CANTIDAD_PARA_IGUALAR = int((y_train == 0).sum() - (y_train == 1).sum())
CANTIDAD_SINTETICOS_INICIAL = CANTIDAD_PARA_IGUALAR
X_sinteticos = crear_morosos_sinteticos(
    X_train_preparado,
    y_train,
    CANTIDAD_SINTETICOS_INICIAL,
)

morosos_reales = y_train.to_numpy() == 1
normales_reales = y_train.to_numpy() == 0

# Los sintéticos se crean en la escala numérica que usa KNN.
# Para graficarlos, volvemos a las unidades originales de ingresos y edad.
escalador_ajustado = modelo_base.named_steps["escalador"]
X_train_para_grafico = escalador_ajustado.inverse_transform(X_train_preparado)
X_sinteticos_para_grafico = escalador_ajustado.inverse_transform(X_sinteticos)
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(
    X_train_para_grafico[normales_reales, 0],
    X_train_para_grafico[normales_reales, 1],
    alpha=0.25,
    s=20,
    label="no moroso real",
)
ax.scatter(
    X_train_para_grafico[morosos_reales, 0],
    X_train_para_grafico[morosos_reales, 1],
    alpha=0.9,
    s=35,
    label="moroso real",
)
ax.scatter(
    X_sinteticos_para_grafico[:, 0],
    X_sinteticos_para_grafico[:, 1],
    alpha=0.35,
    s=18,
    marker="x",
    label="moroso sintético",
)
ax.set_xscale("log")
ax.set_title("Proyección de los sintéticos sobre ingresos y edad")
ax.set_xlabel("ingresos")
ax.set_ylabel("edad (años)")
ax.legend()
plt.show()

print("Morosos reales en train:", int(y_train.sum()))
print("Morosos sintéticos creados:", len(X_sinteticos))
print("Sintéticos necesarios para llegar a 50/50:", CANTIDAD_PARA_IGUALAR)


In [ ]:
def entrenar_y_comparar(cantidad_sinteticos, vecinos=7, ruido=0.05):
    """Compara KNN con el mismo test antes y después del balanceo."""
    X_sinteticos = crear_morosos_sinteticos(
        X_train_preparado,
        y_train,
        cantidad_sinteticos,
        ruido,
    )
    y_sinteticos = np.ones(len(X_sinteticos), dtype=int)

    X_train_balanceado = np.vstack([X_train_preparado, X_sinteticos])
    y_train_balanceado = np.concatenate([y_train.to_numpy(), y_sinteticos])

    knn_base = KNeighborsClassifier(n_neighbors=vecinos)
    knn_balanceado = KNeighborsClassifier(n_neighbors=vecinos)
    knn_base.fit(X_train_preparado, y_train)
    knn_balanceado.fit(X_train_balanceado, y_train_balanceado)

    pred_base = knn_base.predict(X_test_preparado)
    pred_balanceado = knn_balanceado.predict(X_test_preparado)

    tabla = pd.DataFrame(
        [
            calcular_metricas(y_test, pred_base),
            calcular_metricas(y_test, pred_balanceado),
        ],
        index=["KNN sin balancear", "KNN con sintéticos"],
    )
    return tabla, pred_base, pred_balanceado


resultados, _, y_pred_balanceado = entrenar_y_comparar(
    CANTIDAD_SINTETICOS_INICIAL,
    vecinos=VECINOS_BASE,
)
display(resultados.style.format({
    "accuracy": "{:.1%}",
    "precision": "{:.1%}",
    "recall": "{:.1%}",
    "F1": "{:.1%}",
}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_base,
    display_labels=["no moroso", "moroso"],
    colorbar=False,
    cmap="Oranges",
    ax=axes[0],
)
axes[0].set_title(
    f"Sin balancear — recall {resultados.loc['KNN sin balancear', 'recall']:.1%}"
)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred_balanceado,
    display_labels=["no moroso", "moroso"],
    colorbar=False,
    cmap="Greens",
    ax=axes[1],
)
axes[1].set_title(
    f"Con sintéticos — recall {resultados.loc['KNN con sintéticos', 'recall']:.1%}"
)

plt.suptitle("Filas: respuesta real; columnas: respuesta del modelo", y=1.02)
for ax in axes:
    ax.set_xlabel("Predicho")
    ax.set_ylabel("Real")
plt.tight_layout()
plt.show()


### Qué cambió

Con las once variables seleccionadas, el KNN base ya clasifica bien. El balanceo completo reduce todavía más los falsos negativos y mejora recall y F1, aunque puede sumar algunas falsas alarmas.

El resultado es distinto al experimento anterior porque el modelo ahora recibe más información útil. Esto también es una lección: balancear no compensa variables insuficientes; primero necesitamos una representación razonable del problema.

La elección final depende de los costos:

- si es grave dejar pasar un moroso, priorizar recall puede tener sentido;
- si cada revisión humana cuesta tiempo, hay que vigilar precision;
- el test real, separado desde el principio, permite comparar sin autoengañarnos.


---
## Actividad guiada — cambiar una decisión

Modificá solo estas tres variables y ejecutá la celda:

- VECINOS: cantidad de vecinos que KNN consulta;
- CANTIDAD_SINTETICOS: cantidad de morosos nuevos en train;
- RUIDO_SINTETICO: variación alrededor de los puntos intermedios.

Probá de a una opción:

- cantidad igual a 0: ¿qué diferencia queda?
- cantidad igual a 100 y luego CANTIDAD_PARA_IGUALAR: ¿qué cambia al balancear parcialmente o por completo?
- vecinos igual a 3 y luego 25: ¿cómo cambia el sesgo local?
- ruido igual a 0.50: ¿las filas nuevas siguen pareciendo razonables?

La meta no es obtener una cifra perfecta. Es explicar qué pasó con falsos negativos, falsos positivos, precision, recall y la cantidad de casos marcados.


In [ ]:
# CAMBIÁ SOLO ESTAS TRES VARIABLES
VECINOS = 7
CANTIDAD_SINTETICOS = CANTIDAD_SINTETICOS_INICIAL
RUIDO_SINTETICO = 0.05

resultados_actividad, _, _ = entrenar_y_comparar(
    cantidad_sinteticos=CANTIDAD_SINTETICOS,
    vecinos=VECINOS,
    ruido=RUIDO_SINTETICO,
)

display(resultados_actividad.style.format({
    "accuracy": "{:.1%}",
    "precision": "{:.1%}",
    "recall": "{:.1%}",
    "F1": "{:.1%}",
}))


---
## 5. Gestión de problemas comunes

| Problema | Riesgo | Qué hacemos |
|---|---|---|
| Mirar solo accuracy | Oculta falsos negativos de la clase minoritaria | Revisar matriz, precision, recall y F1 |
| Usar test durante la preparación | Fuga de información y evaluación optimista | Ajustar imputación y escala solo con train |
| Balancear antes del split | Filtra ejemplos del test hacia train | Separar primero; sintetizar solo train |
| Generar solicitudes imposibles | El modelo aprende un mundo falso | Validar las combinaciones con el dominio |
| Usar ID como variable | El modelo memoriza identificadores | Excluir columnas que solo identifican filas |
| Codificar texto sin entenderlo | Agrega señales difíciles de interpretar | Dejarlo para una clase posterior y documentarlo |

El balanceo es una hipótesis de mejora, no una garantía. Los datos reales y representativos deciden si sirve.

## Síntesis

- Un proyecto de ML empieza explorando los datos.
- El EDA puede revelar faltantes, escalas y desbalance de clases.
- KNN compara distancias; por eso el preprocesamiento importa.
- Una accuracy aceptable puede convivir con muchos falsos negativos.
- La matriz de confusión muestra qué clase está siendo favorecida.
- Los datos sintéticos pueden balancear train, pero nunca deben contaminar test.
- Mejorar un modelo significa medir trade-offs, no perseguir un único número.

## Comprobación final

1. ¿Qué problema reveló la exploración de moroso?
2. ¿Por qué KNN sin balancear favorecía la clase mayoritaria?
3. ¿Por qué imputación y escala se ajustan solo con train?
4. ¿Qué costo puede tener subir recall mediante datos sintéticos?
